# RAG Avançado com Qdrant + Maritaca

Este notebook implementa três melhorias sobre o RAG Naive:

| Etapa | Técnica | O que faz |
|---|---|---|
| **Pré-recuperação** | Multi-Query | Gera variações da pergunta para buscar mais documentos relevantes |
| **Pós-recuperação** | Compressão contextual | Filtra partes irrelevantes antes de enviar para a LLM |
| **Re-ranqueamento** | CrossEncoder | Reordena os resultados por relevância real (não só similaridade de vetor) |

> **Pré-requisito:** o Qdrant precisa estar rodando e a coleção `imdb_top_1000_rag` já precisa estar populada (rode o notebook 01 primeiro).

## Célula 1 — Instalar dependências extras

O notebook base já instalou `sentence-transformers`. Precisamos garantir que o CrossEncoder está disponível (já vem junto com a biblioteca).

In [3]:
# Instala as mesmas dependências do notebook 01, caso ainda não estejam instaladas.
!pip install -q sentence-transformers qdrant-client openai python-dotenv

## Célula 2 — Imports

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

# CrossEncoder: modelo que avalia (pergunta, documento) juntos — mais preciso que embedding.
# Diferença do bi-encoder: o bi-encoder codifica query e doc separadamente;
# o cross-encoder lê os dois juntos e dá uma nota de relevância.
from sentence_transformers import CrossEncoder

print("Imports realizados com sucesso.")

c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports realizados com sucesso.


## Célula 3 — Configurações

Altere `CSV_PATH` para o caminho correto no seu computador.

In [2]:
# ─────────────────────────────────────────────
# ALTERE AQUI se necessário
# ─────────────────────────────────────────────
COLLECTION_NAME   = "imdb_top_1000_rag"          # mesma coleção do notebook 01
QDRANT_URL        = "http://localhost:6333"
EMBEDDING_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
MARITACA_MODEL    = "sabiazinho-4"
MARITACA_BASE_URL = "https://chat.maritaca.ai/api"

# ─────────────────────────────────────────────
# Parâmetros do RAG Avançado
# ─────────────────────────────────────────────
TOP_K_INICIAL   = 10   # quantos documentos recuperar por query (antes do re-rank)
TOP_N_FINAL     = 3    # quantos documentos usar após o re-ranking
N_QUERIES       = 3    # quantas variações de pergunta gerar no pré-recuperação

print("Configurações carregadas.")
print(f"  TOP_K_INICIAL  : {TOP_K_INICIAL}")
print(f"  TOP_N_FINAL    : {TOP_N_FINAL}")
print(f"  N_QUERIES      : {N_QUERIES}")

Configurações carregadas.
  TOP_K_INICIAL  : 10
  TOP_N_FINAL    : 3
  N_QUERIES      : 3


## Célula 4 — Carregar chave e criar clientes

In [5]:
# Carrega o arquivo .env com a chave da Maritaca.
load_dotenv()
MARITACA_API_KEY = os.getenv("MARITACA_API_KEY")

if not MARITACA_API_KEY:
    raise ValueError("Chave MARITACA_API_KEY não encontrada no .env")

# Cria o cliente do Qdrant.
qdrant_client = QdrantClient(url=QDRANT_URL)

# Cria o cliente da Maritaca (compatível com OpenAI).
maritaca_client = OpenAI(
    api_key=MARITACA_API_KEY,
    base_url=MARITACA_BASE_URL,
)

# Carrega o modelo de embedding (bi-encoder — rápido, para busca inicial).
print("Carregando modelo de embedding...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"  Modelo carregado: {EMBEDDING_MODEL}")

# Verifica se a coleção já existe.
collections = [c.name for c in qdrant_client.get_collections().collections]
if COLLECTION_NAME not in collections:
    raise RuntimeError(
        f"Coleção '{COLLECTION_NAME}' não encontrada no Qdrant.\n"
        "Execute o notebook 01 primeiro para popular a base."
    )

print(f"  Coleção '{COLLECTION_NAME}' encontrada no Qdrant. Pronto!")

Carregando modelo de embedding...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2416.95it/s]


  Modelo carregado: sentence-transformers/all-MiniLM-L6-v2
  Coleção 'imdb_top_1000_rag' encontrada no Qdrant. Pronto!


## Célula 5 — Carregar o CrossEncoder (Re-ranqueamento)

### Por que CrossEncoder e não o mesmo modelo de embedding?

O bi-encoder (all-MiniLM) codifica query e documento **separadamente** → rápido, mas menos preciso.

O cross-encoder codifica `[query] [SEP] [documento]` **juntos** → mais lento, mas muito mais preciso.

Estratégia clássica: bi-encoder para recuperar top-K (rápido), cross-encoder para re-ranquear (preciso).

In [6]:
# Carrega o CrossEncoder. Este modelo foi treinado para ranking de relevância
# em pares (query, passagem). É leve e funciona bem em CPU.
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print("Carregando CrossEncoder (re-ranqueador)...")
reranker = CrossEncoder(RERANKER_MODEL)
print(f"  Modelo carregado: {RERANKER_MODEL}")

# Teste rápido para confirmar que o re-ranqueador funciona.
test_pairs = [
    ("crime movies", "A film about organized crime and the mafia."),
    ("crime movies", "A romantic comedy set in Paris."),
]
test_scores = reranker.predict(test_pairs)
print(f"\nTeste do re-ranqueador:")
print(f"  Pair 1 (relevante)  : score = {test_scores[0]:.4f}")
print(f"  Pair 2 (irrelevante): score = {test_scores[1]:.4f}")
print("  (esperado: pair 1 com score maior)")

Carregando CrossEncoder (re-ranqueador)...


c:\Users\Usuario\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8073.95it/s]


  Modelo carregado: cross-encoder/ms-marco-MiniLM-L-6-v2

Teste do re-ranqueador:
  Pair 1 (relevante)  : score = 2.7868
  Pair 2 (irrelevante): score = -9.9434
  (esperado: pair 1 com score maior)


---
## ① PRÉ-RECUPERAÇÃO: Multi-Query

**Ideia:** antes de buscar no Qdrant, peça à LLM para gerar N variações da pergunta original.
Faça a busca com cada variação e junte os resultados únicos (deduplicando por ID).

**Por quê?** Uma única frase pode ter viés lexical. Variações cobrem mais ângulos semânticos e recuperam documentos que a query original perderia.

In [7]:
def gerar_multiplas_queries(pergunta_original: str, n: int = N_QUERIES) -> list[str]:
    """
    Usa a Maritaca para gerar N variações semânticas da pergunta original.
    Retorna uma lista com a pergunta original + as variações.
    """
    prompt = f"""\
Você é um especialista em recuperação de informação.
Dado a pergunta do usuário abaixo, gere exatamente {n} reformulações alternativas.
Cada reformulação deve ter um ângulo semântico ligeiramente diferente.
Responda APENAS com as {n} reformulações, uma por linha, sem numeração e sem explicações.

Pergunta original: {pergunta_original}
"""
    response = maritaca_client.responses.create(
        model=MARITACA_MODEL,
        input=[{"role": "user", "content": prompt}],
        max_output_tokens=200,
        temperature=0.7,  # temperatura mais alta = variações mais criativas
    )

    # Extrai o texto e separa em linhas.
    texto = response.output[0].content[0].text.strip()
    variações = [linha.strip() for linha in texto.split("\n") if linha.strip()]

    # Garante que a pergunta original está na lista (sempre primeira).
    todas_as_queries = [pergunta_original] + variações[:n]

    return todas_as_queries


# Teste da função.
pergunta_teste = "Quais filmes falam sobre crime ou máfia?"
queries_geradas = gerar_multiplas_queries(pergunta_teste, n=3)

print(f"Pergunta original: {pergunta_teste}")
print(f"\n{len(queries_geradas)} queries geradas:")
for i, q in enumerate(queries_geradas, 1):
    print(f"  {i}. {q}")

Pergunta original: Quais filmes falam sobre crime ou máfia?

4 queries geradas:
  1. Quais filmes falam sobre crime ou máfia?
  2. Quais longas-metragens exploram narrativas centradas em crimes organizados ou no universo da máfia?
  3. Que produções cinematográficas abordam temas relacionados ao crime e à atuação das organizações mafiosas?
  4. Quais filmes se debruçam sobre a vida criminosa ou as dinâmicas das máfias?


In [8]:
def buscar_com_multi_query(
    queries: list[str],
    top_k_por_query: int = TOP_K_INICIAL
) -> list:
    """
    Executa a busca vetorial para cada query e retorna
    uma lista deduplicada de resultados (por ID do ponto).
    Mais queries = mais cobertura semântica.
    """
    vistos = {}  # dicionário para deduplicar por ID

    for q in queries:
        # Gera o embedding da query.
        vetor = embedding_model.encode(
            q,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).tolist()

        # Busca no Qdrant.
        try:
            resultado = qdrant_client.query_points(
                collection_name=COLLECTION_NAME,
                query=vetor,
                limit=top_k_por_query,
                with_payload=True,
            )
            pontos = resultado.points
        except AttributeError:
            pontos = qdrant_client.search(
                collection_name=COLLECTION_NAME,
                query_vector=vetor,
                limit=top_k_por_query,
                with_payload=True,
            )

        # Deduplica: guarda apenas a primeira ocorrência de cada ID.
        for ponto in pontos:
            if ponto.id not in vistos:
                vistos[ponto.id] = ponto

    resultados_unicos = list(vistos.values())
    return resultados_unicos


# Teste: buscar com as queries geradas acima.
resultados_pre = buscar_com_multi_query(queries_geradas, top_k_por_query=TOP_K_INICIAL)

print(f"Documentos únicos recuperados com {len(queries_geradas)} queries: {len(resultados_pre)}")
print("\nTítulos recuperados (antes do re-ranking):")
for i, p in enumerate(resultados_pre, 1):
    print(f"  {i:2}. {p.payload.get('title')} | score vetor: {p.score:.4f}")

Documentos únicos recuperados com 4 queries: 19

Títulos recuperados (antes do re-ranking):
   1. M - Eine Stadt sucht einen Mörder | score vetor: 0.4481
   2. Du rififi chez les hommes | score vetor: 0.4418
   3. Les quatre cents coups | score vetor: 0.4371
   4. À bout de souffle | score vetor: 0.4310
   5. Vivre sa vie: Film en douze tableaux | score vetor: 0.4161
   6. La dolce vita | score vetor: 0.4074
   7. Pulp Fiction | score vetor: 0.4064
   8. The Godfather | score vetor: 0.4058
   9. Scarface: The Shame of the Nation | score vetor: 0.4026
  10. Casino | score vetor: 0.3957
  11. Tropa de Elite 2: O Inimigo Agora é Outro | score vetor: 0.4068
  12. Contratiempo | score vetor: 0.3778
  13. Goodfellas | score vetor: 0.3733
  14. Tropa de Elite | score vetor: 0.3634
  15. Cidade de Deus | score vetor: 0.4685
  16. Dom za vesanje | score vetor: 0.4170
  17. La battaglia di Algeri | score vetor: 0.4158
  18. Nuovo Cinema Paradiso | score vetor: 0.4122
  19. Les diaboliques | scor

---
## ② RE-RANQUEAMENTO

Agora temos um pool de documentos maior (vindos de múltiplas queries).
O CrossEncoder avalia cada par `(pergunta_original, texto_do_filme)` e retorna scores de relevância.
Reordenamos e pegamos apenas os top-N para enviar à LLM.

In [9]:
def reranquear(pergunta: str, documentos: list, top_n: int = TOP_N_FINAL) -> list:
    """
    Usa o CrossEncoder para re-ranquear os documentos pelo par
    (pergunta, texto_do_documento). Retorna os top_n mais relevantes.
    """
    if not documentos:
        return []

    # Monta os pares para o CrossEncoder: [(pergunta, doc1_text), (pergunta, doc2_text), ...]
    pares = [
        (pergunta, doc.payload.get("rag_text", ""))
        for doc in documentos
    ]

    # O CrossEncoder retorna um score de relevância para cada par.
    # Score alto = par relevante; score baixo = par irrelevante.
    scores = reranker.predict(pares)

    # Combina documentos com seus scores e ordena do maior para o menor.
    documentos_com_score = sorted(
        zip(scores, documentos),
        key=lambda x: x[0],
        reverse=True,
    )

    # Retorna apenas os top_n documentos reordenados.
    top_documentos = [doc for _, doc in documentos_com_score[:top_n]]
    top_scores     = [s   for s, _   in documentos_com_score[:top_n]]

    return top_documentos, top_scores


# Aplica o re-ranking nos documentos recuperados.
documentos_reranqueados, scores_reranqueados = reranquear(
    pergunta_teste,
    resultados_pre,
    top_n=TOP_N_FINAL,
)

print(f"Documentos após re-ranking (top {TOP_N_FINAL}):")
print()
for i, (doc, score) in enumerate(zip(documentos_reranqueados, scores_reranqueados), 1):
    titulo = doc.payload.get('title')
    genero = doc.payload.get('genre')
    print(f"  {i}. {titulo}")
    print(f"     Gênero: {genero}")
    print(f"     Score CrossEncoder: {score:.4f}")
    print()

Documentos após re-ranking (top 3):

  1. Casino
     Gênero: Crime, Drama
     Score CrossEncoder: -6.7679

  2. Les quatre cents coups
     Gênero: Crime, Drama
     Score CrossEncoder: -7.0085

  3. Du rififi chez les hommes
     Gênero: Crime, Drama, Thriller
     Score CrossEncoder: -8.6648



---
## ③ PÓS-RECUPERAÇÃO: Compressão Contextual

Antes de montar o contexto final para a LLM, pedimos à própria LLM para extrair
**apenas o trecho relevante** de cada documento.

**Por quê?** Cada documento pode ter partes que não têm nada a ver com a pergunta.
Enviando só o que é relevante: (a) economizamos tokens, (b) reduzimos ruído, 
(c) a LLM responde com mais precisão.

In [10]:
def comprimir_documento(pergunta: str, texto_documento: str) -> str:
    """
    Usa a LLM para extrair apenas a parte do documento
    que é relevante para responder a pergunta.
    Se nada for relevante, retorna uma string vazia.
    """
    prompt = f"""\
Você é um assistente que filtra informações.
Dado o texto de um documento e uma pergunta, extraia SOMENTE as frases do documento
que são diretamente úteis para responder a pergunta.
Se nenhuma parte for relevante, responda exatamente: "IRRELEVANTE"
Não adicione comentários, apenas o trecho extraído (ou a palavra IRRELEVANTE).

Pergunta: {pergunta}

Documento:
{texto_documento}
"""
    response = maritaca_client.responses.create(
        model=MARITACA_MODEL,
        input=[{"role": "user", "content": prompt}],
        max_output_tokens=200,
        temperature=0.0,  # temperatura zero = resposta mais determinística e focada
    )
    trecho = response.output[0].content[0].text.strip()

    # Se a LLM marcar como irrelevante, descartamos.
    if "IRRELEVANTE" in trecho.upper():
        return ""
    return trecho


def comprimir_contexto(pergunta: str, documentos: list) -> str:
    """
    Aplica a compressão em cada documento e monta o contexto final.
    Documentos irrelevantes são descartados.
    """
    blocos = []
    for i, doc in enumerate(documentos, 1):
        texto_original = doc.payload.get("rag_text", "")
        titulo = doc.payload.get("title", "")

        # Comprime o documento para apenas o trecho relevante.
        trecho_relevante = comprimir_documento(pergunta, texto_original)

        if trecho_relevante:
            bloco = f"[Filme {i}: {titulo}]\n{trecho_relevante}"
            blocos.append(bloco)
        else:
            print(f"  ⚠ Documento {i} ({titulo}) descartado como irrelevante.")

    contexto_comprimido = "\n\n---\n\n".join(blocos)
    return contexto_comprimido


# Aplica a compressão nos documentos re-ranqueados.
print(f"Comprimindo contexto para a pergunta: '{pergunta_teste}'")
print()
contexto_final = comprimir_contexto(pergunta_teste, documentos_reranqueados)

print("\nContexto final após compressão:")
print("=" * 70)
print(contexto_final)
print("=" * 70)
print(f"\nTokens aproximados no contexto final: ~{len(contexto_final.split())} palavras")

Comprimindo contexto para a pergunta: 'Quais filmes falam sobre crime ou máfia?'


Contexto final após compressão:
[Filme 1: Casino]
Título: Casino. Ano: 1995. Gênero: Crime, Drama. Sinopse: A tale of greed, deception, money, power, and murder occur between two best friends: a mafia enforcer and a casino executive compete against each other over a gambling empire.

---

[Filme 2: Les quatre cents coups]
Título: Les quatre cents coups. Ano: 1959. Gênero: Crime, Drama. Sinopse: A young boy, left without attention, delves into a life of petty crime.

---

[Filme 3: Du rififi chez les hommes]
Du rififi chez les hommes. Gênero: Crime, Drama, Thriller. Sinopse: Four men plan a technically perfect crime, but the human element intervenes...

Tokens aproximados no contexto final: ~101 palavras


---
## Célula Final — Função `rag_avancado()` completa

Junta os três estágios em uma única função:

In [11]:
def gerar_resposta_final(pergunta: str, contexto: str) -> str:
    """Envia pergunta + contexto comprimido para a Maritaca e retorna a resposta."""
    system_msg = (
        "Você é um assistente especialista em filmes. "
        "Responda em português do Brasil. "
        "Use somente o contexto fornecido. "
        "Se a resposta não estiver no contexto, diga que não encontrou informação suficiente. "
        "Seja claro e direto."
    )
    user_msg = f"""CONTEXTO RECUPERADO E FILTRADO:\n{contexto}\n\nPERGUNTA: {pergunta}"""

    response = maritaca_client.responses.create(
        model=MARITACA_MODEL,
        input=[
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg},
        ],
        max_output_tokens=600,
        temperature=0.2,
    )
    return response.output[0].content[0].text


def rag_avancado(
    pergunta: str,
    n_queries: int = N_QUERIES,
    top_k: int = TOP_K_INICIAL,
    top_n: int = TOP_N_FINAL,
    verbose: bool = True,
) -> str:
    """
    Pipeline completo de RAG Avançado:

    1. PRÉ-RECUPERAÇÃO  → gera n_queries variações da pergunta
    2. RECUPERAÇÃO      → busca top_k documentos por query, deduplica
    3. RE-RANQUEAMENTO  → CrossEncoder reordena e seleciona top_n
    4. PÓS-RECUPERAÇÃO  → compressão contextual filtra ruído
    5. GERAÇÃO          → Maritaca gera a resposta final
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"PERGUNTA: {pergunta}")
        print(f"{'='*60}")

    # ── ETAPA 1: Pré-recuperação (Multi-Query) ──────────────────
    if verbose:
        print("\n[1/4] Pré-recuperação: gerando múltiplas queries...")
    queries = gerar_multiplas_queries(pergunta, n=n_queries)
    if verbose:
        for i, q in enumerate(queries, 1):
            print(f"   {i}. {q}")

    # ── ETAPA 2: Recuperação com deduplicação ───────────────────
    if verbose:
        print(f"\n[2/4] Recuperação: buscando top-{top_k} por query...")
    documentos_recuperados = buscar_com_multi_query(queries, top_k_por_query=top_k)
    if verbose:
        print(f"   {len(documentos_recuperados)} documentos únicos recuperados.")

    # ── ETAPA 3: Re-ranqueamento ────────────────────────────────
    if verbose:
        print(f"\n[3/4] Re-ranqueamento: selecionando top-{top_n}...")
    docs_reranqueados, scores_rr = reranquear(pergunta, documentos_recuperados, top_n=top_n)
    if verbose:
        for i, (doc, sc) in enumerate(zip(docs_reranqueados, scores_rr), 1):
            print(f"   {i}. {doc.payload.get('title')} | CrossEncoder: {sc:.4f}")

    # ── ETAPA 4: Pós-recuperação (Compressão) ───────────────────
    if verbose:
        print("\n[4/4] Pós-recuperação: comprimindo contexto...")
    contexto = comprimir_contexto(pergunta, docs_reranqueados)

    if not contexto.strip():
        return "Não encontrei informação suficiente nos documentos recuperados."

    # ── ETAPA 5: Geração ────────────────────────────────────────
    resposta = gerar_resposta_final(pergunta, contexto)

    if verbose:
        print("\nRESPOSTA FINAL:")
        print("-" * 60)
        print(resposta)
        print("-" * 60)

    return resposta


# Teste completo!
resposta = rag_avancado("Quais filmes falam sobre crime ou máfia?")


PERGUNTA: Quais filmes falam sobre crime ou máfia?

[1/4] Pré-recuperação: gerando múltiplas queries...
   1. Quais filmes falam sobre crime ou máfia?
   2. Quais produções cinematográficas abordam temas relacionados à criminalidade e à máfia?
   3. Que longas-metragens exploram a vida criminosa e o universo das organizações mafiosas?
   4. Quais obras do cinema retratam ações de gangues ou o funcionamento da máfia?

[2/4] Recuperação: buscando top-10 por query...
   23 documentos únicos recuperados.

[3/4] Re-ranqueamento: selecionando top-3...
   1. Casino | CrossEncoder: -6.7679
   2. Les quatre cents coups | CrossEncoder: -7.0085
   3. Scarface | CrossEncoder: -8.5694

[4/4] Pós-recuperação: comprimindo contexto...

RESPOSTA FINAL:
------------------------------------------------------------
Os filmes que falam sobre crime ou máfia são:

- Casino (1995): trata de poder, ganância, assassinato e máfia, com um enfoque em um enforcador da máfia e um executivo de cassino.
- Scarface (1

## Loop Interativo

In [ ]:
print("Chat RAG Avançado iniciado. Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Pergunta: ").strip()

    if not pergunta:
        print("Digite uma pergunta válida.")
        continue

    if pergunta.lower() in ["sair", "exit", "quit"]:
        print("Encerrando. Até mais!")
        break

    rag_avancado(pergunta)

Chat RAG Avançado iniciado. Digite 'sair' para encerrar.


PERGUNTA: quais filmes de comedia

[1/4] Pré-recuperação: gerando múltiplas queries...
   1. quais filmes de comedia
   2. Quais são os melhores filmes de comédia já produzidos?
   3. Que títulos de filmes cômicos você recomenda assistir?
   4. Como classificar os filmes de comédia por popularidade ou crítica?

[2/4] Recuperação: buscando top-10 por query...
   27 documentos únicos recuperados.

[3/4] Re-ranqueamento: selecionando top-3...
   1. Vivre sa vie: Film en douze tableaux | CrossEncoder: -10.2143
   2. Cidade de Deus | CrossEncoder: -10.3169
   3. Nuovo Cinema Paradiso | CrossEncoder: -10.6619

[4/4] Pós-recuperação: comprimindo contexto...
  ⚠ Documento 1 (Vivre sa vie: Film en douze tableaux) descartado como irrelevante.
  ⚠ Documento 2 (Cidade de Deus) descartado como irrelevante.
  ⚠ Documento 3 (Nuovo Cinema Paradiso) descartado como irrelevante.


---
## Comparação: RAG Naive vs RAG Avançado

Execute esta célula para comparar os dois pipelines lado a lado.

In [ ]:
def rag_naive(pergunta: str, top_k: int = 5) -> str:
    """Versão simplificada do RAG Naive (como no notebook 01)."""
    # Embedding direto da pergunta.
    vetor = embedding_model.encode(pergunta, convert_to_numpy=True, normalize_embeddings=True).tolist()

    # Busca simples no Qdrant.
    try:
        resultado = qdrant_client.query_points(collection_name=COLLECTION_NAME, query=vetor, limit=top_k, with_payload=True)
        pontos = resultado.points
    except AttributeError:
        pontos = qdrant_client.search(collection_name=COLLECTION_NAME, query_vector=vetor, limit=top_k, with_payload=True)

    # Monta contexto sem compressão.
    blocos = [f"[Doc {i}]\n{p.payload.get('rag_text', '')}" for i, p in enumerate(pontos, 1)]
    contexto = "\n\n".join(blocos)

    return gerar_resposta_final(pergunta, contexto)


pergunta_comparacao = "Filmes de aventura com personagens heroicos"

print("NAIVE RAG:")
print("=" * 60)
resposta_naive = rag_naive(pergunta_comparacao)
print(resposta_naive)

print("\n\nRAG AVANÇADO:")
resposta_avancado = rag_avancado(pergunta_comparacao)